In [1]:
# add src3 to sys.path for imports
import sys
from pathlib import Path
sys.path.append(str(Path().resolve().parent / "src3"))

In [2]:
import numpy as np
from scipy.io.wavfile import write as wav_write

from utils import save, load, append, symlink
from process_inputs import clean_shifts, fft_shifts, recover_audio, process_fft

In [3]:
BASE_DIR = Path('/home/ethantu/workspace/good-vibrations/data')
sample_id, input_id = '000432', '000432'

sample_dir = BASE_DIR / f'samples/{sample_id}'
speckle_shifts_path = sample_dir / 'inputs/speckle_shifts.npz'

fs = 2500.0
recovered_audio_sr = 22050

# Code

In [4]:
raw_shifts = load(speckle_shifts_path, keys='shifts')[None]  # (L,T,C) -> (1,L,T,C)
raw_shifts.shape

(1, 100, 9000, 2)

In [5]:
cleaned_shifts = clean_shifts(raw_shifts, fs)
save({'clean_shifts': cleaned_shifts}, sample_dir / 'inputs/03_clean_speckle_shifts.npz')
cleaned_shifts.shape

(1, 100, 9000, 2)

In [6]:
fft, freqs, n_samples = fft_shifts(cleaned_shifts, fs)
save({'fft': fft, 'freqs': freqs, 'n_samples': n_samples}, sample_dir / 'inputs/04_clean_speckle_shifts.npz')
fft.shape, freqs.shape, n_samples

((1, 100, 3421, 2), (3421,), 9000)

In [7]:
audio = recover_audio(fft, n_samples, fs, recovered_audio_sr)
save((audio, recovered_audio_sr), sample_dir / 'inputs/05_recovered_audio.wav')
symlink(sample_dir / 'inputs/05_recovered_audio.wav', sample_dir / 'recovered_audio.wav')
load(sample_dir / 'inputs/05_recovered_audio.wav')

In [8]:
processed_fft = process_fft(fft)
save(processed_fft, sample_dir / "inputs/06_processed_fft.npy")
symlink(sample_dir / "inputs/06_processed_fft.npy", sample_dir / 'y.npy')
processed_fft.shape

(1, 100, 13, 256, 2)

# Compare our pipeline against HF reference artifacts

In [9]:
import os, json, torch
import sys
sys.path.insert(0, str(Path().resolve().parent / "src2"))
sys.path.insert(0, str(Path().resolve().parent))   # so "src2.helpers" resolves as a package
from dataset import process_fft as process_fft_ds
from huggingface_hub import snapshot_download
from scipy.io.wavfile import read as wav_read

HF_REPO   = 'eturok-weizmann/laser-vibrations'
HF_SAMPLE_ID = 432   # <-- change this to compare a different sample

# Download just the metadata index to find artifact paths
meta_snap = snapshot_download(HF_REPO, repo_type='dataset', allow_patterns=['data/metadata.jsonl'])
with open(f'{meta_snap}/data/metadata.jsonl') as f:
    rows = [json.loads(l) for l in f]
row      = next(r for r in rows if r['sample_id'] == HF_SAMPLE_ID)
manifest = json.loads(row['manifest'])
arts     = manifest['artifacts']

# Download only the three artifacts we need
patterns = [arts['speckle_shifts_clean'], arts['speckle_shifts_fft'], arts['speckle_shifts_ifft_audio']]
snap     = snapshot_download(HF_REPO, repo_type='dataset', allow_patterns=patterns)

hf_clean = np.load(os.path.join(snap, arts['speckle_shifts_clean']))['shifts_clean']   # (L,T,C) float32
hf_fft   = np.load(os.path.join(snap, arts['speckle_shifts_fft']))['fft']              # (L,F,C) complex64
hf_sr, hf_audio = wav_read(os.path.join(snap, arts['speckle_shifts_ifft_audio']))      # int16

print(f"HF sample_id={HF_SAMPLE_ID}")
print(f"  shifts_clean:     {hf_clean.shape}  {hf_clean.dtype}")
print(f"  shifts_fft:       {hf_fft.shape}  {hf_fft.dtype}")
print(f"  ifft_audio:       {hf_audio.shape}  {hf_audio.dtype}  sr={hf_sr}")

/home/ethantu/workspace/good-vibrations/.venv/lib/python3.12/site-packages/torch/cuda/__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

HF sample_id=432
  shifts_clean:     (100, 9000, 2)  float32
  shifts_fft:       (100, 3421, 2)  complex64
  ifft_audio:       (79380,)  int16  sr=22050


In [10]:
def compare(name, ours, ref, atol):
    ours_f = ours.astype(np.float64)
    ref_f  = ref.astype(np.float64)
    max_err  = float(np.max(np.abs(ours_f - ref_f)))
    mean_err = float(np.mean(np.abs(ours_f - ref_f)))
    match = max_err <= atol
    print(f"{name}")
    print(f"  ours {ours.shape} {ours.dtype}  |  ref {ref.shape} {ref.dtype}")
    print(f"  max |diff|={max_err:.4e}  mean |diff|={mean_err:.4e}  match(atol={atol:.0e}): {match}\n")

# 1. cleaned shifts — drop our batch dim to match HF's (L,T,C)
compare("shifts_clean", cleaned_shifts[0].astype(np.float32), hf_clean, atol=1e-6)

# 2. recovered audio
compare("ifft_audio (int16 counts)", audio, hf_audio, atol=1)

# 3. processed fft — use dataset.py's process_fft on the HF fft (batch size 1)
#    dataset.py unfold gives (1,L,P,C,PS); permute to match our (1,L,P,PS,C)
hf_fft_t     = torch.from_numpy(hf_fft).unsqueeze(0)
hf_processed = process_fft_ds(hf_fft_t, patch_size=256, signal_mode='magnitude',
                               normalize_mode='std-sample', speakers=['0001'], verbose=False)
hf_processed = hf_processed.permute(0, 1, 2, 4, 3).numpy()
compare("processed_fft", processed_fft, hf_processed, atol=1e-6)

shifts_clean
  ours (100, 9000, 2) float32  |  ref (100, 9000, 2) float32
  max |diff|=2.3842e-07  mean |diff|=2.3030e-09  match(atol=1e-06): True

ifft_audio (int16 counts)
  ours (79380,) int16  |  ref (79380,) int16
  max |diff|=1.0000e+00  mean |diff|=1.2346e-03  match(atol=1e+00): True

processed_fft
  ours (1, 100, 13, 256, 2) float32  |  ref (1, 100, 13, 256, 2) float32
  max |diff|=1.9073e-06  mean |diff|=2.2810e-08  match(atol=1e-06): False

